In [ ]:
# This example illustrates a simple parallel non-llm based workflow to understand how parallel workflows can be created using LangGraph.
# The workflow is to calculate Strike Rate, Baoundry per Ball, and Runs in Boundry percentage for a cricket player based on the input data. 
# The workflow will take the input data, calculate the required metrics in parallel, aggregate the results in Summary node, and then return the final result.
# input data is a dictionary with the following keys: 'runs_scored', 'boundaries', 'sixes', 'balls_played'
# Output data is a dictionary with the following keys: 'strike_rate', 'boundary_per_ball', 'runs_in_boundary_percentage'

# NOTE ABOUT STATE UPDATES IN LANGGRAPH:
# In this parallel workflow, each node returns only a partial dictionary update, not the full state.
# LangGraph validates returned keys against CricketStatsState and merges valid keys into shared state.
# Example: return {'strike_rate': value} updates only strike_rate in the global state.
# If a key is not part of CricketStatsState, LangGraph raises a state/update error.
# If parallel nodes write the same key, a reducer/merge strategy is required to resolve conflicts.

In [1]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict

In [2]:
# Define a state
class CricketStatsState(TypedDict):
    runs_scored: int
    boundaries: int
    sixes: int
    balls_played: int
    strike_rate: float
    boundary_per_ball: float
    runs_in_boundary_percentage: float
    summary: str

In [42]:
# Write functions to calculate the required metrics 
# 1: Calculate Strike Rate, 2: Calculate Boundary per Ball, 3: Calculate Runs in Boundary Percentage, 4: Aggregate the results in Summary node
# Calculate Strike Rate
def calculate_strike_rate(state: CricketStatsState):
    strike_rate = (state['runs_scored'] / state['balls_played']) * 100
    return {'strike_rate': strike_rate}

# 2: Calculate Boundary per Ball
def calculate_boundary_per_ball(state: CricketStatsState):
    boundary_per_ball = (state['balls_played'] / (state['boundaries'] + state['sixes'])) if (state['boundaries'] + state['sixes']) > 0 else 0
    return {'boundary_per_ball': boundary_per_ball}

# 3: Calculate Runs in Boundary Percentage
def calculate_runs_in_boundary_percentage(state: CricketStatsState):
    runs_in_boundary_percentage = ((state['boundaries'] * 4 + state['sixes'] * 6) / state['runs_scored']) * 100 if state['runs_scored'] > 0 else 0
    return {'runs_in_boundary_percentage': runs_in_boundary_percentage}

# 4: Aggregate the results in Summary node
def aggregate_summary(state: CricketStatsState):
    summary = f"\nStrike Rate: {state['strike_rate']:.2f}, \nBoundary per Ball: {state['boundary_per_ball']:.2f}, \nRuns in Boundary Percentage: {state['runs_in_boundary_percentage']:.2f}"
    return {'summary': summary}

In [43]:
# Crate a state graph
graph = StateGraph(CricketStatsState )
print(graph.nodes)

{}


In [44]:
# Add nodes to the graph (safe to rerun)
if 'Calculate_Strike_Rate' not in graph.nodes:
    graph.add_node('Calculate_Strike_Rate', calculate_strike_rate)
if 'Calculate_Boundary_per_Ball' not in graph.nodes:
    graph.add_node('Calculate_Boundary_per_Ball', calculate_boundary_per_ball)
if 'Calculate_Runs_in_Boundary_Percentage' not in graph.nodes:
    graph.add_node('Calculate_Runs_in_Boundary_Percentage', calculate_runs_in_boundary_percentage)
if 'Aggregate_Summary' not in graph.nodes:
    graph.add_node('Aggregate_Summary', aggregate_summary)

In [45]:
# Add edges to the graph
print(graph.edges)
graph.add_edge(START, 'Calculate_Strike_Rate')
graph.add_edge(START, 'Calculate_Boundary_per_Ball')
graph.add_edge(START, 'Calculate_Runs_in_Boundary_Percentage')
graph.add_edge('Calculate_Strike_Rate', 'Aggregate_Summary')
graph.add_edge('Calculate_Boundary_per_Ball', 'Aggregate_Summary')
graph.add_edge('Calculate_Runs_in_Boundary_Percentage', 'Aggregate_Summary')
graph.add_edge('Aggregate_Summary', END)
print(graph.edges)


set()
{('Calculate_Strike_Rate', 'Aggregate_Summary'), ('Calculate_Boundary_per_Ball', 'Aggregate_Summary'), ('__start__', 'Calculate_Strike_Rate'), ('__start__', 'Calculate_Runs_in_Boundary_Percentage'), ('Aggregate_Summary', '__end__'), ('Calculate_Runs_in_Boundary_Percentage', 'Aggregate_Summary'), ('__start__', 'Calculate_Boundary_per_Ball')}


In [46]:
# Compile the graph
compiled_workflow = graph.compile()

In [47]:
# Define the input

input_data = {
    'runs_scored': 94,
    'boundaries': 10,
    'sixes': 8,
    'balls_played': 29,
}

In [50]:
# Execute the workflow
result = compiled_workflow.invoke(input_data)

In [51]:
# Print Results
print("Full merged state:", result)
print("Players Stats Summary:", result['summary'])

Full merged state: {'runs_scored': 94, 'boundaries': 10, 'sixes': 8, 'balls_played': 29, 'strike_rate': 324.13793103448273, 'boundary_per_ball': 1.6111111111111112, 'runs_in_boundary_percentage': 93.61702127659575, 'summary': '\nStrike Rate: 324.14, \nBoundary per Ball: 1.61, \nRuns in Boundary Percentage: 93.62'}
Players Stats Summary: 
Strike Rate: 324.14, 
Boundary per Ball: 1.61, 
Runs in Boundary Percentage: 93.62
